# Model testing — données à unité constante et facteurs publics

La préparation privée reprend les renommages et les fonctions de `unite_constante.ipynb` : appariement sur `sPropCode` + `sUnitCode`, filtre strict `0,5 < gap_years < 2,5`, loyers effectifs positifs et croissance annualisée.

- `private_leases` : tous les baux, avec les colonnes renommées.
- `df_private` : paires comparables en loyer effectif, avec les détails du logement, du bail actuel et du précédent, `Market`, `Year`, `type_bail` et la hausse annualisée.
- `df_public` : les trois facteurs publics retenus par marché et année.
- `df_final` : jointure des paires privées et des facteurs publics; une ligne par paire.

`effective_growth_pct` est la cible principale; `growth_pct` conserve le nom utilisé dans le notebook source. Les premiers baux et les paires exclues par les filtres ne sont pas dans `df_private`. Toutes les données sont préparées en mémoire, sans export CSV.

In [ ]:
"""Extraction reproductible des cinq fichiers SCHL, sans dépendance Excel supplémentaire."""
from pathlib import Path
import posixpath
import re
import xml.etree.ElementTree as ET
from zipfile import ZipFile
import pandas as pd

NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}
REL = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
CMHC_ALIASES = {"Montréal (RMR)": "Montréal", "Ottawa-Gatineau (RMR) (ON)": "Ottawa — partie Ontario",
           "Ottawa-Gatineau (RMR) (partie ON)": "Ottawa — partie Ontario"}

def read_workbook(path):
    """Lit les valeurs enregistrées des cellules, y compris les chaînes partagées."""
    with ZipFile(path) as archive:
        strings = []
        if "xl/sharedStrings.xml" in archive.namelist():
            strings = ["".join(node.itertext()) for node in ET.fromstring(
                archive.read("xl/sharedStrings.xml")).findall("m:si", NS)]
        targets = {r.attrib["Id"]: r.attrib["Target"] for r in ET.fromstring(
            archive.read("xl/_rels/workbook.xml.rels"))}
        sheets = {}
        for sheet in ET.fromstring(archive.read("xl/workbook.xml")).findall("m:sheets/m:sheet", NS):
            target = targets[sheet.attrib[f"{{{REL}}}id"]]
            target = target.lstrip("/") if target.startswith("/") else posixpath.normpath(posixpath.join("xl", target))
            rows = []
            for row in ET.fromstring(archive.read(target)).findall("m:sheetData/m:row", NS):
                cells = {}
                for cell in row.findall("m:c", NS):
                    value = cell.find("m:v", NS)
                    if cell.get("t") == "inlineStr":
                        text = "".join(cell.find("m:is", NS).itertext())
                    elif value is None:
                        continue
                    else:
                        text = strings[int(value.text)] if cell.get("t") == "s" else value.text
                    column = re.sub(r"\d+", "", cell.attrib["r"])
                    cells[column] = text
                rows.append((int(row.attrib["r"]), cells))
            sheets[sheet.attrib["name"]] = rows
        return sheets

def number(value):
    """Conserve les données supprimées/non disponibles comme valeurs manquantes."""
    if value is None:
        return float("nan")
    cleaned = re.sub(r"[\s\u00a0\u202f]", "", str(value)).replace(",", ".")
    return float(cleaned) if re.fullmatch(r"-?\d+(\.\d+)?", cleaned) else float("nan")

def extract_public_data(raw_dir="raw_public_data"):
    records, audit = [], []
    for year in range(2021, 2026):
        path = Path(raw_dir) / f"rmr-canada-{year}-fr.xlsx"
        sheets = read_workbook(path)
        name1 = next(name for name in sheets if re.fullmatch(r"Table(?:au)? 1\.0", name))
        name4 = next(name for name in sheets if re.fullmatch(r"Table(?:au)? 4\.1", name))
        rows1, rows4 = dict(sheets[name1]), dict(sheets[name4])
        # Vérifier les en-têtes avant d'appliquer les colonnes identifiées dans les cinq fichiers.
        assert "inoccupation" in rows1[6]["B"].lower()
        assert any(word in rows1[6]["G"].lower() for word in ["rotation", "roulement"])
        assert "deux chambres" in rows1[6]["L"].lower()
        assert "univers" in rows4[6]["L"].lower()
        period = rows1[7]["D"]
        assert str(year) in period or str(year)[-2:] in period, (path, period)
        universes = {CMHC_ALIASES[cells["A"]]: (row, cells) for row, cells in sheets[name4]
                     if cells.get("A") in CMHC_ALIASES}
        for row, cells in sheets[name1]:
            label = cells.get("A")
            if label not in CMHC_ALIASES:
                continue
            market = CMHC_ALIASES[label]
            universe_row, universe_cells = universes[market]
            # D/I/N = année du fichier dans 1.0 ; N = univers ELL, pas univers des copropriétés.
            record = {"year": year, "market": market,
                      "vacancy_rate": number(cells.get("D")),
                      "turnover_rate": number(cells.get("I")),
                      "rent": number(cells.get("N")),
                      "rental_universe": number(universe_cells.get("N"))}
            # Les valeurs ELL du tableau 4.1 doivent concorder avec celles du tableau 1.0.
            assert record["vacancy_rate"] == number(universe_cells.get("D")), (path, market)
            assert record["rent"] == number(universe_cells.get("I")), (path, market)
            records.append(record)
            for variable, table, source_row, column, source_cells, quality in [
                ("vacancy_rate", name1, row, "D", cells, "E"),
                ("turnover_rate", name1, row, "I", cells, "J"),
                ("rent", name1, row, "N", cells, "O"),
                ("rental_universe", name4, universe_row, "N", universe_cells, None),
            ]:
                audit.append({"year": year, "market": market, "variable": variable,
                              "source_file": path.name, "source_sheet": table,
                              "source_cell": f"{column}{source_row}",
                              "raw_value": source_cells.get(column),
                              "quality_code": source_cells.get(quality) if quality else None})
    df = pd.DataFrame(records).sort_values(["market", "year"]).reset_index(drop=True)
    df["rental_universe"] = df.rental_universe.astype("Int64")
    assert len(df) == 10 and not df.duplicated(["year", "market"]).any()
    assert df[["vacancy_rate", "turnover_rate", "rent", "rental_universe"]].notna().all().all()
    assert df.vacancy_rate.between(0, 100).all() and df.turnover_rate.between(0, 100).all()
    assert df.rent.gt(0).all() and df.rental_universe.gt(0).all()
    return df, pd.DataFrame(audit)



In [ ]:
from pathlib import Path
import re
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

RAW_PUBLIC = Path("raw_public_data")
PRIVATE = Path("private_data")
pd.set_option("display.max_columns", 80)
MARKETS = ["Montréal", "Ottawa — partie Ontario"]

def slug(value):
    text = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", text).strip("_")

def read_public_csv(path):
    with path.open(encoding="utf-8-sig") as stream:
        sep = ";" if ";" in stream.readline() else ","
    return pd.read_csv(path, sep=sep, encoding="utf-8-sig").rename(columns={
        "PÉRIODE DE RÉFÉRENCE": "period", "REF_DATE": "period",
        "GÉO": "geo", "GEO": "geo", "VALEUR": "value", "VALUE": "value",
        "UNITÉ DE MESURE": "unit", "UOM": "unit",
        "FACTEUR SCALAIRE": "scalar", "SCALAR_FACTOR": "scalar",
    })

specs = [
    ("IPC.csv", "cpi", "Produits et groupes de produits"),
    ("msrche-du-travail.csv", "labour", "Caractéristiques de la population active"),
]
frames = {}
for filename, family, dimension in specs:
    frame = read_public_csv(RAW_PUBLIC / filename)
    frame["value"] = pd.to_numeric(frame.value, errors="raise")
    frame["Year"] = pd.to_datetime(frame.period, format="%Y-%m").dt.year
    frames[family] = frame


## 1. Les trois facteurs publics retenus

`general_cpi` est l'IPC général national moyen, base 2002=100. `vacancy_rate` est le taux d'inoccupation SCHL. `unemployement_rate` est le taux de chômage moyen : Montréal pour le Québec, Toronto comme substitut pour Ottawa. Les périodes sources sont conservées séparément.

In [ ]:
cmhc, cmhc_sources = extract_public_data(RAW_PUBLIC)
public_columns = ["general_cpi", "vacancy_rate", "unemployement_rate"]
df_public = cmhc.rename(columns={"market": "Market", "year": "Year"}).copy()

cpi = frames["cpi"].loc[frames["cpi"].geo.eq("Canada") & frames["cpi"]["Produits et groupes de produits"].eq("Ensemble")].copy()
assert not cpi.duplicated(["Year", "period"]).any()
assert cpi.unit.eq("2002=100").all()
cpi_yearly = cpi.groupby("Year").value.mean().rename("general_cpi").reset_index()
df_public = df_public.merge(cpi_yearly, on="Year", how="left", validate="many_to_one")

labour = frames["labour"].loc[frames["labour"]["Caractéristiques de la population active"].eq("Taux de chômage")].copy()
labour["Market"] = labour.geo.map({"Montréal, Québec": "Montréal", "Toronto, Ontario": "Ottawa — partie Ontario"})
labour = labour.loc[labour.Market.notna()]
assert not labour.duplicated(["Market", "period"]).any()
assert labour.unit.eq("Pourcent").all()
unemployment = labour.groupby(["Market", "Year"]).value.mean().rename("unemployement_rate").reset_index()
df_public = df_public.merge(unemployment, on=["Market", "Year"], how="left", validate="one_to_one")
df_public = df_public[["Market", "Year"] + public_columns].sort_values(["Market", "Year"]).reset_index(drop=True)
assert not df_public.duplicated(["Market", "Year"]).any()
assert df_public[public_columns].notna().all().all()

source_rows = []
for _, row in df_public[["Market", "Year"]].iterrows():
    year, market = row.Year, row.Market
    for family, frame in [("general_cpi", cpi.loc[cpi.Year.eq(year)]),
                          ("unemployement_rate", labour.loc[labour.Year.eq(year) & labour.Market.eq(market)])]:
        source_rows.append({"Market": market, "Year": year, "factor": family,
                            "source_geo": ", ".join(frame.geo.unique()),
                            "months_available": frame.period.nunique(),
                            "reference_periods": ", ".join(sorted(frame.period.unique()))})
df_public_sources = pd.DataFrame(source_rows)
assert df_public_sources.months_available.eq(12).all(), "Année incomplète : examiner avant utilisation."
display(df_public)
display(df_public_sources)


### Définitions

- `general_cpi` : moyenne annuelle de l'indice général national; ce n'est pas un taux d'inflation.
- `vacancy_rate` : inoccupation régionale en %.
- `unemployement_rate` : taux de chômage moyen en %, Toronto étant le substitut ontarien demandé.

La jointure utilise `Market` et `Year`. Les modèles décalent les facteurs d'un an pour la prévision; les dates de publication restent à vérifier.

In [ ]:
df_public_status = pd.DataFrame({
    "factor": public_columns,
    "unit": ["indice 2002=100", "%", "%"],
    "available_rows": [df_public[col].notna().sum() for col in public_columns],
    "total_rows": len(df_public),
})
display(df_public_status)

In [ ]:
# Méthode et renommages repris de unite_constante.ipynb.
UNIT_KEY = ["sPropCode", "sUnitCode"]
MIN_GAP_YEARS = 0.5
MAX_GAP_YEARS = 2.5
DAYS_PER_YEAR = 365.25
PCT = 100
PREVIOUS_LEASE_COLUMNS = ["contract_rent", "effective_rent", "lease_start", "lease_term_months"]
LEASE_COLUMN_RENAMES = {
    "sRent": "contract_rent",
    "sRentEffective": "effective_rent",
    "sConcession": "has_concession",
    "sRenewal": "is_renewal",
    "sTermSeq": "unit_lease_rank",
    "sTermMonths": "lease_term_months",
    "sLeaseFrom": "lease_start",
    "sLeaseTo": "lease_end",
    "sSignDate": "sign_date",
    "sAvailable": "lease_start_duplicate",
    "sLeaseTerm": "lease_term_label_unreliable",
}

def pair_consecutive_leases(lease_df, unit_key):
    """Associe chaque bail au bail précédent de la même unité (colonnes préfixées par `prev_`).

    Retourne uniquement les baux qui ont un bail précédent.
    """
    ordered = lease_df.sort_values(unit_key + ["lease_start"]).copy()
    previous = ordered.groupby(unit_key)[PREVIOUS_LEASE_COLUMNS].shift(1).add_prefix("prev_")
    paired = pd.concat([ordered, previous], axis=1).dropna(subset=["prev_lease_start"])
    paired["gap_years"] = (paired["lease_start"] - paired["prev_lease_start"]).dt.days / DAYS_PER_YEAR
    return paired

def filter_comparable_pairs(paired, price_col, verbose=False):
    """Garde les paires dont l'écart est comparable et dont les deux loyers sont positifs."""
    prev_price_col = f"prev_{price_col}"
    steps = [
        ("écart > MIN_GAP_YEARS", paired["gap_years"] > MIN_GAP_YEARS),
        ("écart < MAX_GAP_YEARS", paired["gap_years"] < MAX_GAP_YEARS),
        ("loyers positifs", (paired[prev_price_col] > 0) & (paired[price_col] > 0)),
    ]
    keep = pd.Series(True, index=paired.index)
    funnel = [("paires appariées", len(paired))]
    for label, condition in steps:
        keep &= condition
        funnel.append((label, int(keep.sum())))
    if verbose:
        funnel_table = pd.DataFrame(funnel, columns=["étape", "paires restantes"])
        funnel_table["paires retirées"] = -funnel_table["paires restantes"].diff().fillna(0).astype(int)
        print(funnel_table.to_string(index=False))
    return paired[keep].copy()

def annualized_growth_pct(paired, price_col):
    """Croissance annualisée (%) entre le bail précédent et le bail courant."""
    price_ratio = paired[price_col] / paired[f"prev_{price_col}"]
    return (price_ratio ** (1 / paired["gap_years"]) - 1) * PCT

def same_unit_growth(lease_df, price_col):
    """Croissance annualisée du loyer à unité constante : apparier, filtrer, annualiser."""
    paired = pair_consecutive_leases(lease_df, UNIT_KEY)
    comparable = filter_comparable_pairs(paired, price_col)
    comparable["growth_pct"] = annualized_growth_pct(comparable, price_col)
    return comparable

def median_growth_table(pairs, by):
    return (pairs.groupby(by)["growth_pct"]
            .agg(pairs="size", median="median")
            .round(2))

private_leases = pd.read_csv("private_data/equinoxe_lease_history.csv", dtype={"sPropCode": str, "sUnitCode": str})
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    private_leases[col] = pd.to_datetime(private_leases[col], errors="raise")
private_leases = private_leases.rename(columns=LEASE_COLUMN_RENAMES)
for col in ["contract_rent", "effective_rent", "lease_term_months", "is_renewal"]:
    private_leases[col] = pd.to_numeric(private_leases[col], errors="raise")
private_leases["lease_start_year"] = private_leases.lease_start.dt.year
assert private_leases[UNIT_KEY].notna().all().all()
assert not private_leases.duplicated(UNIT_KEY + ["lease_start"]).any(), "Baux ambigus : vérifier les doublons avant appariement."

# La cible retenue est la croissance du loyer effectif, comme dans effective_pairs.
effective_pairs = same_unit_growth(private_leases, "effective_rent")
contract_pairs = same_unit_growth(private_leases, "contract_rent")
df_private = effective_pairs.copy()
df_private["Market"] = df_private.sState.map({"Quebec": "Montréal", "Ontario": "Ottawa — partie Ontario"})
df_private["Year"] = df_private.lease_start_year.astype("Int64")
df_private["type_bail"] = df_private.is_renewal.map({0: "Relocation", 1: "Renouvellement"})
df_private["effective_growth_pct"] = df_private.growth_pct
# La hausse contractuelle est un diagnostic complémentaire sur ces mêmes paires.
contract_valid = df_private.contract_rent.gt(0) & df_private.prev_contract_rent.gt(0)
df_private["contract_growth_pct"] = annualized_growth_pct(df_private, "contract_rent").where(contract_valid)
leading = ["sPropCode", "sUnitCode", "Market", "Year", "type_bail", "effective_rent", "prev_effective_rent", "gap_years", "effective_growth_pct"]
df_private = df_private[leading + [c for c in df_private if c not in leading]]
assert np.isfinite(df_private.effective_growth_pct).all()
assert not df_public.duplicated(["Market", "Year"]).any()
df_final = df_private.merge(df_public, on=["Market", "Year"], how="left", validate="many_to_one")
assert len(df_final) == len(df_private)
display(df_private[leading].head(12).round(3))
display(df_final.head())
print(f"Baux : {len(private_leases):,}; paires comparables : {len(df_private):,}; lignes jointes : {len(df_final):,}")


## Vérifier la couverture

Les paires sont rattachées à l'année du nouveau bail. Les paires antérieures aux sources publiques restent présentes avec des facteurs publics manquants. On affiche les effectifs et la médiane de la croissance effective par marché/année/type de bail.

Cette jointure contemporaine sert à explorer les relations. Pour une régression de prévision, ne pas utiliser `effective_rent`, `growth_pct`, `contract_growth_pct` ni les autres informations du nouveau bail comme prédicteurs automatiques de `effective_growth_pct`. Vérifier la disponibilité temporelle des facteurs publics et séparer les années lors de la validation.

In [ ]:
coverage = df_final.groupby(["Market", "Year", "type_bail"], dropna=False).agg(
    nombre_paires=("sPropCode", "size"),
    hausse_effective_mediane_pct=("effective_growth_pct", "median"),
    paires_avec_taux_inoccupation=("vacancy_rate", "count"),
)
display(coverage.round(3))
display(df_public_status)
display(df_final.loc[df_final.Market.eq("Ottawa — partie Ontario")].head())

## Jointure finale avec le nombre de chambres

`df_private` contient les onze attributs privés retenus, dont `bedrooms` (provenant de `sBeds`) et `has_concession`. `df_final` ajoute seulement l'IPC général, l'inoccupation et le chômage sur `Year` et `Market`. Les autres détails restent dans `df_private_details`.

In [ ]:
private_attributes = [
    "sPropCode", "sUnitCode", "Market", "Year", "type_bail",
    "effective_rent", "prev_effective_rent", "gap_years", "effective_growth_pct",
    "has_concession", "bedrooms",
]
public_attributes = ["general_cpi", "vacancy_rate", "unemployement_rate"]
df_private_details = df_private.copy()
df_private_details["bedrooms"] = pd.to_numeric(df_private_details.sBeds, errors="raise")
df_private = df_private_details[private_attributes].copy()
df_public_retained = df_public[["Year", "Market"] + public_attributes].copy()
assert df_private.shape[1] == 11
assert not df_public_retained.duplicated(["Year", "Market"]).any()
df_final = df_private.merge(
    df_public_retained,
    on=["Year", "Market"],
    how="left",
    validate="many_to_one",
)
assert len(df_final) == len(df_private), "La jointure a multiplié les paires."
assert df_final.columns.tolist() == private_attributes + public_attributes
print(f"Privé : {df_private.shape}; jointure finale : {df_final.shape} (11 colonnes privées + 3 publiques).")
display(df_private.head())
display(df_final.head(20))
